#### 1. Hãy kết nối hai bảng trên theo những cách sau: 

In [11]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("student.db")
cursor = conn.cursor()

In [12]:
cursor.executescript("""
DROP TABLE IF EXISTS student;
DROP TABLE IF EXISTS course;

CREATE TABLE student (
    student_id INTEGER,
    name TEXT,
    class TEXT,
    course_id INTEGER,
    score REAL
);

CREATE TABLE course (
    id INTEGER,
    course_name TEXT
);
""")

cursor.executemany("INSERT INTO student VALUES (?, ?, ?, ?, ?)", [
(1,"Nguyen Minh Hoang","May Tinh",12,6.7),
(2,"Tran Thi Lan","Kinh Te",34,9.2),
(3,"Pham Van Nam","Toan Tin",None,7.9),
(4,"Le Thanh Huyen","Toan Tin",20,7.2),
(5,"Vu Quoc Anh","May Tinh",24,8.0),
(6,"Dang Thuy Linh","May Tinh",24,5.5),
(7,"Bui Tien Dung","Kinh Te",34,9.2),
(8,"Ho Ngoc Mai","Toan Tin",20,8.8),
(9,"Duong Huu Phuc","Kinh Te",None,7.2),
(10,"Cao Thi Hanh","May Tinh",None,7.0)
])

cursor.executemany("INSERT INTO course VALUES (?, ?)", [
(12,"Giai tich"),
(34,"Thong ke"),
(26,"Tin hoc")
])

conn.commit()

In [13]:
def run_query(q):
    display(pd.read_sql_query(q, conn))

#### YÊU CẦU 1: KẾT NỐI BẢNG

##### Tích Descartes

In [14]:
run_query("""
          SELECT *
FROM student s, course c
""")

,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,12,Giai tich
1,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,34,Thong ke
2,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,26,Tin hoc
3,2,Tran Thi Lan,Kinh Te,34.0,9.2,12,Giai tich
4,2,Tran Thi Lan,Kinh Te,34.0,9.2,34,Thong ke
5,2,Tran Thi Lan,Kinh Te,34.0,9.2,26,Tin hoc
6,3,Pham Van Nam,Toan Tin,NaN,7.9,12,Giai tich
7,3,Pham Van Nam,Toan Tin,NaN,7.9,34,Thong ke
8,3,Pham Van Nam,Toan Tin,NaN,7.9,26,Tin hoc
9,4,Le Thanh Huyen,Toan Tin,20.0,7.2,12,Giai tich


##### INNER JOIN

In [15]:
run_query("""
SELECT s.*, c.course_name
FROM student s
INNER JOIN course c ON s.course_id = c.id
""")

,student_id,name,class,course_id,score,course_name
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,Giai tich
1,2,Tran Thi Lan,Kinh Te,34,9.2,Thong ke
2,7,Bui Tien Dung,Kinh Te,34,9.2,Thong ke


##### LEFT JOIN

In [16]:
run_query("""
SELECT s.*, c.course_name
FROM student s
LEFT JOIN course c ON s.course_id = c.id
""")

,student_id,name,class,course_id,score,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,Giai tich
1,2,Tran Thi Lan,Kinh Te,34.0,9.2,Thong ke
2,3,Pham Van Nam,Toan Tin,NaN,7.9,None
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2,None
4,5,Vu Quoc Anh,May Tinh,24.0,8.0,None
5,6,Dang Thuy Linh,May Tinh,24.0,5.5,None
6,7,Bui Tien Dung,Kinh Te,34.0,9.2,Thong ke
7,8,Ho Ngoc Mai,Toan Tin,20.0,8.8,None
8,9,Duong Huu Phuc,Kinh Te,NaN,7.2,None
9,10,Cao Thi Hanh,May Tinh,NaN,7.0,None


##### RIGHT JOIN

In [17]:
run_query("""
SELECT s.*, c.course_name
FROM course c
LEFT JOIN student s ON s.course_id = c.id
""")

,student_id,name,class,course_id,score,course_name
0,1.0,Nguyen Minh Hoang,May Tinh,12.0,6.7,Giai tich
1,2.0,Tran Thi Lan,Kinh Te,34.0,9.2,Thong ke
2,7.0,Bui Tien Dung,Kinh Te,34.0,9.2,Thong ke
3,NaN,None,None,NaN,NaN,Tin hoc


##### FULL JOIN 

In [18]:
run_query("""
SELECT s.*, c.course_name
FROM student s
LEFT JOIN course c ON s.course_id = c.id
UNION
SELECT s.*, c.course_name
FROM course c
LEFT JOIN student s ON s.course_id = c.id
""")

,student_id,name,class,course_id,score,course_name
0,NaN,None,None,NaN,NaN,Tin hoc
1,1.0,Nguyen Minh Hoang,May Tinh,12.0,6.7,Giai tich
2,2.0,Tran Thi Lan,Kinh Te,34.0,9.2,Thong ke
3,3.0,Pham Van Nam,Toan Tin,NaN,7.9,None
4,4.0,Le Thanh Huyen,Toan Tin,20.0,7.2,None
5,5.0,Vu Quoc Anh,May Tinh,24.0,8.0,None
6,6.0,Dang Thuy Linh,May Tinh,24.0,5.5,None
7,7.0,Bui Tien Dung,Kinh Te,34.0,9.2,Thong ke
8,8.0,Ho Ngoc Mai,Toan Tin,20.0,8.8,None
9,9.0,Duong Huu Phuc,Kinh Te,NaN,7.2,None


#### YÊU CẦU 2: XỬ LÝ DỮ LIỆU

##### Xóa course_id không hợp lệ

In [19]:
cursor.execute("""
DELETE FROM student
WHERE course_id NOT IN (SELECT id FROM course)
""")
conn.commit()

##### Tổng số sinh viên + điểm TB

In [20]:
run_query("""
SELECT COUNT(*) AS tong_sv, AVG(score) AS diem_tb
FROM student
""")

,tong_sv,diem_tb
0,6,7.866667


##### Theo từng môn

In [21]:
run_query("""
SELECT c.course_name, COUNT(*) AS so_sv, AVG(score) AS diem_tb
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
""")

,course_name,so_sv,diem_tb
0,Giai tich,1,6.7
1,Thong ke,2,9.2


##### Phân loại

In [22]:
run_query("""
SELECT c.course_name,
       AVG(score) AS diem_tb,
       CASE 
           WHEN AVG(score) > 9 THEN 'Xuat sac'
           WHEN AVG(score) BETWEEN 6 AND 8.9 THEN 'Tot'
           ELSE 'Kem'
       END AS xep_loai
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
""")

,course_name,diem_tb,xep_loai
0,Giai tich,6.7,Tot
1,Thong ke,9.2,Xuat sac


#### YÊU CẦU 3: XẾP HẠNG

##### Top 3 cao nhất

In [23]:
run_query("""
SELECT * FROM (
    SELECT *, RANK() OVER (ORDER BY score DESC) r
    FROM student
)
WHERE r <= 3
""")

,student_id,name,class,course_id,score,r
0,2,Tran Thi Lan,Kinh Te,34.0,9.2,1
1,7,Bui Tien Dung,Kinh Te,34.0,9.2,1
2,3,Pham Van Nam,Toan Tin,NaN,7.9,3


##### Top 3 thấp nhất

In [24]:
run_query("""
SELECT * FROM (
    SELECT *, RANK() OVER (ORDER BY score ASC) r
    FROM student
)
WHERE r <= 3
""")

,student_id,name,class,course_id,score,r
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,1
1,10,Cao Thi Hanh,May Tinh,NaN,7.0,2
2,9,Duong Huu Phuc,Kinh Te,NaN,7.2,3


##### Theo lớp

In [25]:
run_query("""
SELECT *,
RANK() OVER (PARTITION BY class ORDER BY score DESC) AS rank_class
FROM student
""")

,student_id,name,class,course_id,score,rank_class
0,2,Tran Thi Lan,Kinh Te,34.0,9.2,1
1,7,Bui Tien Dung,Kinh Te,34.0,9.2,1
2,9,Duong Huu Phuc,Kinh Te,NaN,7.2,3
3,10,Cao Thi Hanh,May Tinh,NaN,7.0,1
4,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,2
5,3,Pham Van Nam,Toan Tin,NaN,7.9,1


##### Theo môn

In [26]:
run_query("""
SELECT *,
RANK() OVER (PARTITION BY course_id ORDER BY score DESC) AS rank_course
FROM student
""")

,student_id,name,class,course_id,score,rank_course
0,3,Pham Van Nam,Toan Tin,NaN,7.9,1
1,9,Duong Huu Phuc,Kinh Te,NaN,7.2,2
2,10,Cao Thi Hanh,May Tinh,NaN,7.0,3
3,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,1
4,2,Tran Thi Lan,Kinh Te,34.0,9.2,1
5,7,Bui Tien Dung,Kinh Te,34.0,9.2,1


#### YÊU CẦU 4: GRADUATION DATE

##### Thêm cột

In [27]:
cursor.execute("ALTER TABLE student ADD COLUMN graduation_date TEXT")
conn.commit()

##### Cập nhật ngày tốt nghiệp

In [28]:
cursor.execute("""
UPDATE student
SET graduation_date = datetime(
    'now',
    '+' || (
        SELECT COUNT(*) 
        FROM student s2
        WHERE s2.score > student.score
    ) || ' days'
)
""")
conn.commit()

##### Kiểm tra

In [29]:
run_query("""
SELECT name, score, graduation_date
FROM student
ORDER BY score DESC
""")

,name,score,graduation_date
0,Tran Thi Lan,9.2,2026-03-30 15:59:29
1,Bui Tien Dung,9.2,2026-03-30 15:59:29
2,Pham Van Nam,7.9,2026-04-01 15:59:29
3,Duong Huu Phuc,7.2,2026-04-02 15:59:29
4,Cao Thi Hanh,7.0,2026-04-03 15:59:29
5,Nguyen Minh Hoang,6.7,2026-04-04 15:59:29
